# Reproducing Wang et al. (2025) from DANDI:000363

Paper. Wang ZA, Kurgyis B, Chen S, Kang B, Chen F, Liu Y, Liu D, Svoboda K, Li N, Druckmann S
(2025). Brain-wide analysis reveals movement encoding structured across and within brain areas.
*Nature Neuroscience*. [doi:10.1038/s41593-025-02114-x](https://doi.org/10.1038/s41593-025-02114-x).
The claims file (`claims/wang-2025.yaml`) was written from the version of record (main text,
legends, Methods, Extended Data legends) and the Supplementary Information.

Dataset. [DANDI:000363](https://dandiarchive.org/dandiset/000363/0.230822.0128), Mesoscale
Activity Map Dataset, version 0.230822.0128. This is the version the paper cites (its data
reference, doi:10.48324/dandi.000363/0.230822.0128), so it is pinned. The most recent published
version, 0.231012.2129 (the one the code's README cites), holds the same 174 sessions, most of
them in much larger files (65.7 TB in total, against 53.6 GB); the files renamed `+image` add the
raw video. The raw video, from which the paper's autoencoder embedding and end-to-end networks are
computed, is only in the later version. The difference, a check that the units, trials and markers
of four sessions are the same in both, and the cost of reading the video are shown in the data
check.

What the dataset holds. 174 sessions of 28 mice in the delayed-response task: sorted units with
CCF annotations, the trial table, behavioural events (licks, epochs), and three side-camera
DeepLabCut markers (jaw, nose, tongue, with a likelihood) per frame. It does not hold the paper's
16-dimensional autoencoder embedding, nor the trained networks.

Outside inputs. From the authors' processed-data deposit, Zenodo doi:10.5281/zenodo.16740783
(named in the claims file), two small members read by HTTP range requests: `area2anno.json`, the
code's grouping of CCF annotation names into areas, and the session names of `cal4cat-delay.pickle`,
the authors' Fig. 6 and 7 file, which name the 106 sessions the paper used. Both are tested by
the link claim `link-authors-deposit`.

| Claim | Tested here |
|---|---|
| link-authors-deposit (link, added by this attempt) | tested |
| data-scale | tested |
| fig6e-marker-clustering | tested |
| fig2b-medulla-vs-midbrain, fig2b-area-ordering, edfig3-epochs, fig2e-method-improvement, fig2-ordering-across-methods, edfig1-reliable-unpredicted | blocked: need the autoencoder embedding and end-to-end networks (too_large) |
| fig3-thalamus-nonuniform, fig3e-nearest-neighbor, fig3f-thalamic-nuclei, thalamus-ev-not-rate, supfig2-noise-injection | blocked: same |
| fig4b-medulla-leads, edfig6-offset-ev-correlation, fig5a-somatosensory-vs-alm, fig5b-po-vs-vm, fig5c-pretectal-vs-scm, fig5-significant-offsets, fig4e-alm-depth | blocked: same |
| fig6a-choice-from-video, fig6d-mouse-clustering, fig6-marker-vs-embedding-decoding, edfig7-performance-correlation | blocked: same |
| fig7b-jaw-differs, fig7c-modulation-nonuniform, fig7d-alm-midbrain-vs-medulla, edfig10-subtraction-consistent, supp-session-independent-autoencoder | blocked: same, or depend on the embedding decoder |

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import collections
import json
import multiprocessing as mp
import resource
import time
import warnings
import zlib

import numpy as np
import pandas as pd
from scipy import stats

import mapvideo as mv

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)

PAPER = "wang-2025"
SEED_FIG6E = zlib.crc32(f"{PAPER}/fig6e-marker-clustering".encode())
N_SESSIONS = int(os.environ.get("N_SESSIONS", "0"))  # 0 = all; a subset only while developing

started = time.time()
assets = mv.list_assets()
print(f"dandiset {mv.DANDISET} version {mv.VERSION}: {len(assets)} assets, "
      f"{sum(a['size'] for a in assets) / 1e9:.1f} GB")
todo = assets[:N_SESSIONS] if N_SESSIONS else assets
with mp.get_context("fork").Pool(4) as pool:
    S = list(pool.imap(mv.session_summary_retry, [(a, SEED_FIG6E) for a in todo], chunksize=1))
failed = [s for s in S if "error" in s]
S = [s for s in S if "error" not in s]
print(f"read and decoded {len(S)} sessions in {(time.time() - started) / 60:.1f} min; failed: {len(failed)}")
for s in failed:
    print("  failed:", s["path"], s["error"])
print("peak memory of this process (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "path": s["path"], "key": s["key"], "subject": s["subject"], "mouse": s["key"].split("_")[0],
    "units": len(s["units"]["anno"]), "trials": s["n_trials"], "included": s["n_included"],
    "correct": s["n_correct"], "left": s["n_left"], "right": s["n_right"],
    "performance": s["performance"], "go_missing": s["n_go_missing"],
    "frames": s["frame_dt"]["n_frames"], "frame_dt_ms": 1000 * s["frame_dt"]["median"],
    "dt_exact": s["frame_dt"]["within_1us"], "video_gaps": s["frame_dt"]["n_gaps"],
    "cameras": len(s["cameras"]), **{f"excl_{k}": v for k, v in s["excluded"].items()},
    "fp_trials": s["fp_trials"], "fp_spikes": s["fp_spikes"], "fp_jaw": s["fp_jaw"]} for s in S])
print("Sessions:", len(info), "; subjects (subject_id):", info.subject.nunique(),
      "; subject descriptions (the authors' mouse names):", info.mouse.nunique())
print("Each subject_id has one description:", bool((info.groupby("subject").mouse.nunique() == 1).all()))
print("\nSessions per mouse:")
print(info.groupby("mouse").size().to_string())
print("\nTrials: total", info.trials.sum(), "; included after the paper's exclusions", info.included.sum(),
      "; correct among included", info.correct.sum())
print("Excluded trials (a trial can have several reasons):",
      {k: int(info[f"excl_{k}"].sum()) for k in ["ignore", "early", "free_water", "auto_water", "photostim"]})
print("Trials without a go cue:", int(info.go_missing.sum()))
print("Correct trials per session: min", info.correct.min(), "median", info.correct.median(), "max", info.correct.max())
print("Performance (correct / included) per session: min %.3f median %.3f max %.3f"
      % (info.performance.min(), info.performance.median(), info.performance.max()))

All 174 sessions have the three side-camera markers. The trial table has 28 subjects, as the paper's
Methods state (28 mice); the files' `subject/description` (for example SC015) is the authors' mouse
name, used below to match the authors' session names. Trials are excluded as the Methods say
(photoinhibition, free water, early licks, no response); `auto_water` trials are counted as free
water.

In [ ]:
# Duplicate recordings: identical trial start times, spike times or jaw traces in two files.
for col in ["fp_trials", "fp_spikes", "fp_jaw"]:
    dup = info[info.duplicated(col, keep=False)]
    print(f"{col}: {info[col].nunique()} distinct among {len(info)} sessions; duplicated:",
          dup[["path", col]].values.tolist() if len(dup) else "none")
print("Sessions with the same mouse and date:", int(info.duplicated("key").sum()))

In [ ]:
# Marker frames and their clock.
print("Marker frame interval (ms): median over sessions", np.round(info.frame_dt_ms.median(), 4),
      "; range", np.round(info.frame_dt_ms.min(), 4), "to", np.round(info.frame_dt_ms.max(), 4))
print("Fraction of within-trial intervals equal to 3.4 ms (to 1 us): min %.4f median %.4f"
      % (info.dt_exact.min(), info.dt_exact.median()))
print("Video gaps (intervals > 10 ms) per session minus trials: median",
      int((info.video_gaps - info.trials).median()), "; range", int((info.video_gaps - info.trials).min()),
      "to", int((info.video_gaps - info.trials).max()))
print("Frames per session: min", info.frames.min(), "median", int(info.frames.median()), "max", info.frames.max())
out = pd.DataFrame([{**{"key": s["key"]}, **{f"{m}_outliers_per_1e4": 1e4 * s["outliers"][m][0] / s["outliers"][m][1]
                                              for m in s["outliers"]}} for s in S])
print("Five-sigma velocity outliers imputed, per 10,000 frames: jaw median %.1f (max %.1f), nose median %.1f (max %.1f)"
      % (out.Jaw_outliers_per_1e4.median(), out.Jaw_outliers_per_1e4.max(),
         out.Nose_outliers_per_1e4.median(), out.Nose_outliers_per_1e4.max()))
print("Correct trials dropped for missing marker frames in the delay epoch: jaw",
      sum(s["n_missing_frames"].get("Jaw", 0) for s in S), "; nose", sum(s["n_missing_frames"].get("Nose", 0) for s in S))
miss = pd.DataFrame([{"key": s["key"], "correct": s["n_correct"], "without_jaw_frames": s["n_missing_frames"].get("Jaw", 0)}
                     for s in S if s["n_missing_frames"].get("Jaw", 0)])
print("Sessions with correct trials lacking marker frames in the delay epoch:")
print(miss.to_string(index=False))

The marker timestamps advance by exactly 3.4 ms within a trial, a frame rate of 294 Hz where
the Methods say 300 Hz; the video is recorded per trial (one gap per trial). The 3.4 ms step is
the stride the Methods give for binning, so the paper worked at the same step. The timestamps are
regular to the microsecond, so they were most likely written from a nominal rate, not measured;
whether they hold the camera's true timing cannot be told from the files.

In [ ]:
# Clock check: the tongue (DeepLabCut likelihood > 0.9) against the trial table's go cue and the
# lick events, true and shifted in time.
rel = np.array(S[0]["tongue_vs_go"][0])
prof = np.array([s["tongue_vs_go"][1] for s in S])
print("Fraction of frames with the tongue visible, in 100 ms windows from the go cue (mean over sessions):")
print(pd.Series(prof.mean(0).round(3), index=np.round(rel, 1)).to_string())
pre, post = prof[:, rel < -0.05].mean(1), prof[:, (rel > 0.05) & (rel < 0.5)].mean(1)
print(f"Per session, 0.1 to 0.5 s after the go cue minus 1 s to 0 before: min {np.min(post - pre):.3f}, "
      f"median {np.median(post - pre):.3f}; sessions where it is not positive: {int(np.sum(post - pre <= 0))}")
lk = pd.DataFrame([s["tongue_at_licks"] for s in S])
print("\nFraction of lick events with the tongue visible within one frame, by shift of the lick times (s):")
print(lk.describe().loc[["min", "50%", "max"]].round(3).to_string())
print("Sessions where the unshifted fraction is below 0.9:", int((lk[0.0] < 0.9).sum()))

In most sessions the tongue appears only after the go cue of the trial table, and it is visible at
nearly every lick event, while lick times shifted by 50 or 100 ms find it much less often. The
sessions where this fails are listed and examined in the next cell. The spike times are not used by
a tested claim except through unit counts, and their clock was not checked here.

In [ ]:
# Sessions where the check fails: scan the shift of the lick times from -1 to 1 s.
flag = info.assign(contrast=post - pre, at_licks=lk[0.0].values)
flag = flag[(flag.contrast <= 0) | (flag.at_licks < 0.9) | (flag.frames < 0.5 * flag.frames.median())]
print(flag[["key", "path", "frames", "trials", "included", "correct", "contrast", "at_licks"]].round(3).to_string(index=False))
by_path = {a["path"]: a for a in assets}
scan = []
for p in flag.path:
    r = mv.lick_alignment(by_path[p])
    best = int(np.nanargmax(r["frac"]))
    scan.append({"key": info.set_index("path").key[p], "licks": r["n_licks"],
                 "included": r["included"], "included_with_frames_at_go": r["included_with_frames"],
                 "frac_at_0": round(float(r["frac"][r["shifts"] == 0][0]), 3),
                 "best_shift_s": float(r["shifts"][best]), "frac_at_best": round(float(r["frac"][best]), 3),
                 "video_span_s": np.round(r["video_span"], 1).tolist(), "trial_span_s": np.round(r["trial_span"], 1).tolist(),
                 "trial_videos": r["segments"], "frames_per_trial_video": r["frames_per_segment"],
                 "videos_with_licks": r["segments_with_licks"], "aligned_gt_0.8": r["segments_aligned"],
                 "misaligned_lt_0.3": r["segments_misaligned"],
                 **{f"stretch_x{k}": round(v, 3) for k, v in r["stretch"].items()}})
scan = pd.DataFrame(scan)
print(scan.T.to_string())
print("Median frames per trial video over all sessions:", int(np.median([s["frame_dt"]["n_frames"] / max(s["frame_dt"]["n_gaps"] + 1, 1) for s in S])))
CLOCK_FLAGGED = set(flag.key)

In seven sessions the check fails, and the table above shows why. In five (SC011 on 24, 25 and 26
February 2019, SC022 on 27 and 28 February 2019) the tongue is visible at 6 to 13% of lick events
as deposited and at no shift of the lick times within 1 s does this reach 0.49, while stretching
each trial's video timestamps by a factor of 2 from its first frame puts the tongue at 99.9 to 100%
of the licks. Their trial videos hold about 940 frames, against a median of 1904 over all
sessions. In these five sessions the frames are 6.8 ms apart, not the 3.4 ms the timestamps give,
so their markers are on a time base half as long as the trial table's; nothing in the files shows
this. In SC026_080719 the tongue is at 60% of lick events: 214 trial videos match the licks and 117
do not, and the session has 539 trial videos for 630 trials, so some videos are likely attached to
the wrong trials (the cause is not established). SC066_041321 has markers for 7 trial videos (49 s)
only. These seven sessions are left out of the main analysis of fig6e; none of them is among the
106 sessions the authors used.

In [ ]:
# Units and their quality labels.
U = pd.concat([pd.DataFrame({"key": s["key"], "mouse": s["key"].split("_")[0], "anno": s["units"]["anno"],
                             "quality": s["units"]["quality"], "classification": s["units"]["classification"],
                             "rate": s["units"]["rate"], "group_location": s["units"]["group_location"]})
               for s in S], ignore_index=True)
print("Units:", len(U))
print("unit_quality:", U.quality.value_counts().to_dict())
print("classification:", U.classification.value_counts().to_dict())
print(pd.crosstab(U.quality, U.classification).to_string())
print("Units without a CCF annotation (empty anno_name):", int((U.anno == "").sum()))
print("avg_firing_rate (Hz): min %.3g median %.3g max %.3g; NaN %d" % (U.rate.min(), U.rate.median(), U.rate.max(), U.rate.isna().sum()))
loc = U.group_location.str.extract(r'"brain_regions": "([^"]*)"')[0]
U["probe_region"] = loc.fillna(U.group_location)
print("Probe targets (electrode group location, brain_regions):")
print(U.probe_region.value_counts().to_string())

In [ ]:
# The later version: what it adds, and what reading its video would cost.
v = mv.video_read_estimate()
print(f"Version {v['version']}: {v['n_assets']} assets, {v['bytes_total'] / 1e12:.1f} TB; "
      f"{v['n_video_assets']} files with video ({v['bytes_video_assets'] / 1e12:.1f} TB)")
print(f"Example {v['example']}: {v['side_series']} side-view series, {v['side_frames']} side-view frames of "
      f"{v['frame_shape']} uint8, read at {v['frames_per_s']:.0f} frames/s")
later = {a["path"].replace("+image", ""): a for a in mv.list_assets(v["version"])}
print("Sessions in both versions, by path without '+image':", sum(a["path"] in later for a in assets), "of", len(assets))
grown = sum(later[a["path"]]["size"] > 10 * a["size"] for a in assets if a["path"] in later)
print("Of these, files more than 10 times larger in the later version:", grown, "; named '+image':",
      sum("+image" in later[a["path"]]["path"] for a in assets if a["path"] in later))
rng_v = np.random.default_rng(zlib.crc32(f"{PAPER}/version-check".encode()))
pick = [assets[i] for i in sorted(rng_v.choice(len(assets), 4, replace=False))]
rows = []
for a in pick:
    old_fp = mv.version_fingerprints(a)
    new_fp = mv.version_fingerprints(later[a["path"]], v["version"])
    rows.append({"path": a["path"].split("/")[1][:40], "later_path_has_image": "+image" in later[a["path"]]["path"],
                 **{k: (old_fp[k] == new_fp[k]) for k in old_fp}})
print("Same values in both versions (4 sessions drawn at random):")
print(pd.DataFrame(rows).to_string(index=False))
hours = v["side_frames"] / v["frames_per_s"] / 3600
print(f"Reading the side view of one session: {hours:.1f} h; of the paper's 106 sessions: {106 * hours:.0f} h, "
      "before any network is trained")

The paper's embedding is computed by a convolutional autoencoder trained on each session's
side-view video, and its end-to-end networks are trained on the frames themselves (Methods,
Convolutional autoencoder; End-to-end learning). The pinned version has no video; the later
version has it, and the estimate above shows that only reading it is far beyond the five-hour run
and the notebook limit of sixty minutes, before training any of the roughly 100 autoencoders and
several hundred end-to-end networks on four CPUs. The per-frame embeddings the authors computed are
not in the Zenodo deposit (its `session_latents.pkl` holds 16 images of 120 by 112 pixels per
session, the decoder's components, not per-frame values) and the Google Drive folder the code's
README links is not pinned. Every claim that needs the embedding or the networks is therefore
blocked as `too_large`.

## link-authors-deposit

> The authors' deposit (Zenodo doi:10.5281/zenodo.16740783) can be linked to the dataset: the CCF
> annotation names of its area grouping (`area2anno.json`) are the dataset's `anno_name` values with
> the commas removed, and the 106 session names of its Fig. 6 and 7 file (`cal4cat-delay.pickle`)
> are sessions of the dataset, named by mouse and date.

In [ ]:
amap = mv.area_map()
dataset_names = set(U.anno[U.anno != ""].str.replace(",", "", regex=False))
rows = []
for area, names in amap.items():
    found = [n for n in names if n in dataset_names]
    rows.append({"area": area, "names_in_map": len(names), "names_found_in_dataset": len(found),
                 "units": int(U.anno.str.replace(",", "", regex=False).isin(names).sum())})
link_areas = pd.DataFrame(rows)
print(link_areas.to_string(index=False))
in_any = set().union(*map(set, amap.values()))
print("\nAreas with at least one dataset annotation:", int((link_areas.names_found_in_dataset > 0).sum()), "of", len(amap))
print("Distinct dataset annotations:", len(dataset_names), "; found in the map:", len(dataset_names & in_any))
print("Map names whose spelling differs from every dataset name only by case or spaces:",
      sorted(n for n in in_any - dataset_names
             if n.lower().replace(" ", "") in {d.lower().replace(" ", "") for d in dataset_names}))

authors = mv.authors_sessions()
by_day = info.groupby("key").path.apply(list).to_dict()
day = {k: "_".join(k.split("_")[:2]) for k in authors}  # SC038_111919_s4 -> SC038_111919
matched = [k for k in authors if day[k] in by_day]
print(f"\nAuthors' Fig. 6/7 sessions: {len(authors)}; matched to a dataset session by mouse and date: {len(matched)}")
print("Not matched:", [k for k in authors if day[k] not in by_day])
print("Mice among the matched sessions:", len({k.split('_')[0] for k in matched}))
same_day = collections.Counter(day[k] for k in matched)
print("Days with more than one authors' session (suffix _s<n>):", {d: n for d, n in same_day.items() if n > 1})
print("Of those, days with as many dataset files:", {d: len(by_day[d]) for d, n in same_day.items() if n > 1})
authors_set = {day[k] for k in matched}
print("Dataset sessions in the authors' set:", int(info.key.isin(authors_set).sum()),
      "; mice:", info[info.key.isin(authors_set)].mouse.nunique(),
      "; dataset sessions not in it:", int((~info.key.isin(authors_set)).sum()))
print("Mice with no session in the authors' set:", sorted(set(info.mouse) - set(info[info.key.isin(authors_set)].mouse)))

**Result: reproduced.** Every one of the 16 areas of the authors' grouping has CCF annotation
names that occur in the dataset, spelled the same once commas are removed; 162 of the dataset's 294
distinct annotations fall in the map, the rest being structures outside its 16 areas (the map has
no isocortex group except the orbital area). All 106 session names of the authors' Fig. 6 and 7 file
match a dataset session by mouse and date; the five days with two sessions (SC038, suffix `_s<n>`)
each have two dataset files, and the 106 sessions come from 21 of the 28 mice. Seven mice (SC011,
SC013, SC015, SC016, SC017, SC022, SC023) have no session in the authors' set.

## data-scale

> The analysis covers more than 50,000 neurons recorded in more than a dozen cortical and
> subcortical structures, simultaneously with high-speed video of orofacial movements.

Criterion: units passing the dataset's quality labels in the sessions with video, before and after
the 2 Hz filter; more than 12 distinct structures at the level of the areas the paper names. The
areas are the 16 of the code's grouping (`area2anno.json`) and ALM (units on probes targeted to
ALM, as the code's `get_single_area_inds` takes them). The rate is the dataset's `avg_firing_rate`
(see the notes); the quality labels are read two ways, `unit_quality` good and `classification`
good.

In [ ]:
U["clean"] = U.anno.str.replace(",", "", regex=False)
for area, names in amap.items():
    U[f"A:{area}"] = U.clean.isin(names)
U["A:ALM"] = U.probe_region.str.contains("ALM")
area_cols = [c for c in U.columns if c.startswith("A:")]
rows = []
for label, sel in [("all units", np.ones(len(U), bool)),
                   ("unit_quality good", U.quality == "good"),
                   ("classification good", U.classification == "good")]:
    for rate_label, r in [("no rate filter", np.ones(len(U), bool)), ("rate > 2 Hz", U.rate > 2)]:
        for sess_label, ss in [("174 sessions", np.ones(len(U), bool)), ("authors' 106", U.key.isin(authors_set))]:
            m = sel & r & ss
            n_areas = int(sum(U.loc[m, c].any() for c in area_cols))
            n_areas100 = int(sum(U.loc[m, c].sum() >= 100 for c in area_cols))
            rows.append({"units": label, "rate": rate_label, "sessions": sess_label, "n": int(m.sum()),
                         "areas_with_units": n_areas, "areas_with_100_units": n_areas100,
                         "distinct_annotations": U.loc[m & (U.anno != ""), "anno"].nunique()})
scale = pd.DataFrame(rows)
print(scale.to_string(index=False))
m = (U.quality == "good")
print("\nUnits per area, unit_quality good, all sessions (an area can contain another):")
print(pd.Series({c[2:]: int(U.loc[m, c].sum()) for c in area_cols}).sort_values(ascending=False).to_string())
print("\nunit_quality good units per mouse: min", U[m].groupby("mouse").size().min(), "max", U[m].groupby("mouse").size().max(),
      "; mice:", U[m].mouse.nunique())

**Result: reproduced.** The 174 sessions, all with video markers, hold 272,227 units; 154,948 have
`unit_quality` good and 69,453 have `classification` good, both above 50,000. With the 2 Hz filter
on the dataset's `avg_firing_rate`, 88,647 and 46,658 remain, so the count is above 50,000 after
the filter under `unit_quality` and below it under `classification`; the criterion holds when
either count exceeds 50,000. The units fall in all 17 areas (the 16 of the authors' grouping and
ALM), more than 12; only units with `classification` good carry a CCF annotation (200,922
unlabelled units have none). Over the authors' 106 sessions, 90,324 units have `unit_quality` good
and 42,259 `classification` good.

## fig6e-marker-clustering

> With single markers the mouse-specific structure of uninstructed movements remains; for some mice
> the nose is more informative than the jaw and for others the jaw more than the nose.

Per session, choice is decoded from the jaw marker alone and from the nose marker alone in the
delay epoch (correct trials, 20-fold stratified cross-validation, L2 logistic regression with the
penalty chosen by nested cross-validation; decoder reading (a), the five-frame window, and (b), one
frame). The session's AUC is the mean over the time bins of the delay epoch (-1.2 to 0 s, every
68 ms). Kept: sessions with jaw or nose AUC of at least 0.65, and mice with two or more such
sessions. Calinski-Harabasz score of the two AUCs with mouse labels against 1,000 random
reassignments of the labels. The main session set is every session of the dataset whose markers
pass the clock check above; all decoded sessions, and the 106 sessions named in the authors' file,
are variants.

In [ ]:
bt = mv.bin_times()
trim = (bt >= -1.05) & (bt <= -0.15)
print("Delay-epoch bins (s from the go cue):", np.round(bt, 3).tolist())
dec = [s for s in S if s["auc"]]
print(f"Sessions decoded: {len(dec)} of {len(S)}; not decoded (fewer than 20 correct trials of a side):",
      [(s['key'], s['n_left'], s['n_right']) for s in S if not s["auc"]])


def session_aucs(reading="a", off=0, window=None):
    w = np.ones(len(bt), bool) if window is None else window
    return pd.DataFrame([{"key": s["key"], "mouse": s["key"].split("_")[0],
                          "jaw": s["auc"][("Jaw", reading, off)][w].mean(),
                          "nose": s["auc"][("Nose", reading, off)][w].mean()} for s in dec])


def clustering(A, rng, thr=0.65, n_null=1000):
    keep = A[(A.jaw >= thr) | (A.nose >= thr)]
    mice = keep.mouse.value_counts()
    keep = keep[keep.mouse.isin(mice[mice >= 2].index)]
    if keep.mouse.nunique() < 2:
        return dict(sessions=len(keep), mice=keep.mouse.nunique(), ch=np.nan, null_mean=np.nan, null_sd=np.nan, p=np.nan,
                    nose_mice=0, jaw_mice=0), keep
    obs, null, p = mv.ch_null(keep[["nose", "jaw"]].values, keep.mouse.values, rng, n_null)
    frac_nose = keep.assign(n=keep.nose > keep.jaw).groupby("mouse").n.mean()
    return dict(sessions=len(keep), mice=keep.mouse.nunique(), ch=obs, null_mean=null.mean(), null_sd=null.std(),
                p=p, nose_mice=int((frac_nose > 0.5).sum()), jaw_mice=int((frac_nose < 0.5).sum())), keep


MAIN = set(info.key) - CLOCK_FLAGGED
print("Main session set: the sessions that pass the clock check,", int(info.key.isin(MAIN).sum()), "of", len(info),
      "(five SC038 days hold two sessions under one mouse-and-date name; both are kept)")
A = session_aucs("a", 0)
A = A[A.key.isin(MAIN)]
print("\nSession AUCs, reading (a), claim's seed: jaw mean %.3f (range %.3f to %.3f), nose mean %.3f (range %.3f to %.3f)"
      % (A.jaw.mean(), A.jaw.min(), A.jaw.max(), A.nose.mean(), A.nose.min(), A.nose.max()))
res_main, keep_main = clustering(A, np.random.default_rng(SEED_FIG6E))
print("Main (reading (a), sessions passing the clock check, threshold 0.65):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in res_main.items()})
print("Paper: CH 12.57, null 1.05 +/- 0.54 s.d., P < 0.001")

In [ ]:
print("Per mouse, kept sessions (main):")
per_mouse = keep_main.groupby("mouse").agg(sessions=("key", "size"), jaw=("jaw", "mean"), nose=("nose", "mean"),
                                           nose_above_jaw=("key", lambda k: int((keep_main.loc[k.index, "nose"] > keep_main.loc[k.index, "jaw"]).sum())))
print(per_mouse.round(3).to_string())
print("\nAll sessions per mouse, before the filter:")
allm = A.groupby("mouse").agg(sessions=("key", "size"), passing=("key", lambda k: int(((A.loc[k.index, "jaw"] >= 0.65) | (A.loc[k.index, "nose"] >= 0.65)).sum())),
                              jaw=("jaw", "mean"), nose=("nose", "mean"))
print(allm.round(3).to_string())

In [ ]:
# Variants: decoder reading, epoch trimming, session set, threshold; each with its own null.
rows = []
auth = authors_set
for reading in ["a", "b"]:
    for wlabel, w in [("full delay", None), ("trimmed -1.05 to -0.15", trim)]:
        Av = session_aucs(reading, 0, w)
        for slabel, sset in [("clock-checked", MAIN), ("all decoded", None), ("authors' 106", auth)]:
            Avs = Av if sset is None else Av[Av.key.isin(sset)]
            for thr in [0.6, 0.65, 0.7]:
                r, _ = clustering(Avs, np.random.default_rng(SEED_FIG6E), thr)
                rows.append({"reading": reading, "window": wlabel, "sessions_set": slabel, "threshold": thr,
                             "n_sessions_decoded": len(Avs), **r})
variants = pd.DataFrame(rows)
print(variants.round(4).to_string(index=False))

In [ ]:
# Seed check: the decoders' splits and the null under the claim's seed and seed + 1000 ... 4000.
rows = []
for off in mv.SEED_OFFSETS:
    Ao = session_aucs("a", off)
    for slabel, sset in [("clock-checked", MAIN), ("authors' 106", auth)]:
        Aos = Ao[Ao.key.isin(sset)]
        r, _ = clustering(Aos, np.random.default_rng(SEED_FIG6E + off))
        rows.append({"seed_offset": off, "sessions_set": slabel, **r})
seeds = pd.DataFrame(rows)
print(seeds.round(4).to_string(index=False))
print("Clock-flagged sessions among the authors' 106:", sorted(CLOCK_FLAGGED & auth))
for slabel in ["clock-checked", "authors' 106"]:
    x = seeds[seeds.sessions_set == slabel]
    print(f"{slabel}: CH from {x.ch.min():.3f} to {x.ch.max():.3f}; p from {x.p.min():.3f} to {x.p.max():.3f}")

**Result: partially reproduced.** Over the 167 sessions that pass the clock check (166 decoded;
SC032_121719 has 8 correct left trials), 55 sessions of
15 mice pass the paper's filter, and their nose and jaw AUCs cluster by mouse: Calinski-Harabasz
score 5.92 against a null of 1.02 ± 0.33 s.d. (paper 1.05 ± 0.54), and no shuffled score reaches it
(p < 0.001). Both kinds of mice occur: 3 mice have the nose above the jaw in most of their kept
sessions and 11 the jaw above the nose. The score is about half the paper's 12.57 and outside its
tolerance (11.31 to 13.83). Over the authors' 106 sessions it is 7.63 (36 sessions, 11 mice), and
over every variant printed (decoder reading, epoch trimming, session set, threshold) it ranges
from 3.61 to 7.90, never within tolerance, with p of at most 0.003. Under the five seeds the score
is 5.92 to 6.53 (authors' 106, 7.63 to 9.22) and p < 0.001 throughout, so the outcome does not
depend on the seed. The mouse-specific structure is present and significant, but weaker than the
paper reports.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

## Blocked claims

Every other claim needs the paper's video embedding, its end-to-end networks, or the choice decoder
run on the embedding (Figs. 6 and 7). None of these is in the pinned version; the raw video is in
the later version, and the estimate under the data check shows that computing them does not fit
(`too_large`). The claims and the reason for each are in the attempt record.

## Summary

| Claim | Outcome |
|---|---|
| link-authors-deposit (link) | reproduced |
| data-scale | reproduced |
| fig6e-marker-clustering | partially reproduced |
| The other 25 claims (Figs. 2 to 7, Extended Data Figs. 1 to 10, Supplementary Discussion) | blocked: too_large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's embedding, its end-to-end networks and
its embedding-based choice decoder could not be computed: the cited version has no video, and the
later version's video would take about 100 hours to read for the paper's sessions before any
network is trained. Only the claims that rest on the markers or on unit counts were tested. The
marker choice decoder of Fig. 6e is not described in the paper and its code is not public; the
claims file's two readings were run (five-frame window and single frame, L2 logistic regression with
a nested choice of penalty) and give the same outcome. AUCs were computed every 68 ms in the delay
epoch, not every frame. The 2 Hz filter uses the dataset's whole-session `avg_firing_rate`, not the
rate over included trials. The main session set for Fig. 6e is every session that passes the clock
check (167), not the paper's 106, which are shown as a variant.

What was found about the dataset. In five sessions (SC011 and SC022, February 2019) the marker
timestamps run twice as fast as the video: the frames are 6.8 ms apart, not 3.4 ms, and only with
the stretch do the tongue markers meet the lick events. In SC026_080719 117 trial videos do not
match the lick events, SC066_041321 has markers for 7 trial videos only, and SC065_050521 lacks
marker frames for 140 of its 354 correct trials. Elsewhere the timestamps
step by exactly 3.4 ms (294 Hz, where the paper says 300 Hz). Only units with `classification`
good have a CCF annotation. The version the paper cites has no video and no embedding; the later
version adds the raw video. The dataset holds 174 sessions of 28 mice, of which the authors used
106 sessions of 21 mice for Figs. 6 and 7, a selection the paper does not describe.